## Import

In [46]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import os
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader

In [47]:
# Custom Dataset for Object Detection
class ObjectDetectionDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.image_files = [f for f in os.listdir(image_dir) if f.endswith(".jpg")]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.image_files[idx])
        label_path = os.path.join(
            self.label_dir, self.image_files[idx].replace(".jpg", ".txt")
        )

        # Load image
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = image.astype(np.float32) / 255.0

        # Load labels (YOLO format)
        bboxes = []
        class_labels = []
        if os.path.exists(label_path):
            with open(label_path, "r") as file:
                for line in file.readlines():
                    parts = list(map(float, line.strip().split()))
                    class_label = int(parts[0])
                    if 1 <= class_label <= 4:  # Ensure only valid labels
                        class_labels.append(class_label)
                        bboxes.append(parts[1:])  # (x, y, w, h) normalized

        bboxes = (
            torch.tensor(bboxes, dtype=torch.float32)
            if bboxes
            else torch.zeros((0, 4), dtype=torch.float32)
        )
        class_labels = (
            torch.tensor(class_labels, dtype=torch.long)
            if class_labels
            else torch.zeros((0,), dtype=torch.long)
        )

        if self.transform:
            image = self.transform(image)

        return image, bboxes, class_labels

In [48]:
# DataLoader Setup
def get_dataloaders(batch_size=16, image_size=224):
    transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Resize((image_size, image_size))]
    )

    train_dataset = ObjectDetectionDataset(
        "data/train/images", "data/train/labels", transform
    )
    val_dataset = ObjectDetectionDataset(
        "data/val/images", "data/val/labels", transform
    )
    test_dataset = ObjectDetectionDataset(
        "data/test/images", "data/test/labels", transform
    )

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn
    )

    return train_loader, val_loader, test_loader


# Collate function to handle variable number of objects per image
def collate_fn(batch):
    images, bboxes, labels = zip(*batch)
    images = torch.stack(images, dim=0)
    return images, bboxes, labels

In [49]:
# Loss Functions
class ObjectDetectionLoss(nn.Module):
    def __init__(self):
        super(ObjectDetectionLoss, self).__init__()
        self.bbox_loss = nn.SmoothL1Loss()
        self.class_loss = nn.CrossEntropyLoss()

    def forward(self, pred_bboxes, pred_classes, target_bboxes, target_classes):
        bbox_loss = self.bbox_loss(pred_bboxes, target_bboxes)
        class_loss = self.class_loss(pred_classes, target_classes)
        return bbox_loss + class_loss

In [50]:
# Model Definition
class CustomObjectDetector(nn.Module):
    def __init__(self, num_classes):
        super(CustomObjectDetector, self).__init__()

        # Backbone (Feature Extractor - ResNet18)
        self.backbone = models.resnet18(pretrained=True)
        self.backbone.fc = nn.Identity()  # Remove ResNet classifier

        # Bounding Box Regression Head (x, y, w, h)
        self.bbox_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 4),  # 4 values (x, y, w, h)
        )

        # Classification Head
        self.class_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),  # Output class probabilities
        )

    def forward(self, x):
        features = self.backbone(x)
        bboxes = self.bbox_head(features)  # Bounding box coordinates
        class_logits = self.class_head(features)  # Class scores
        return bboxes, class_logits

In [51]:
# Training Function
def train(model, dataloader, optimizer, criterion, device, epochs=10):
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for images, target_bboxes, target_classes in dataloader:
            images = images.to(device)
            target_bboxes = [b.to(device) for b in target_bboxes]
            target_classes = [c.to(device) for c in target_classes]

            optimizer.zero_grad()
            pred_bboxes, pred_classes = model(images)

            # Compute loss per batch (assuming a batch of varying object counts)
            loss = sum(
                criterion(
                    pred_bboxes[i], pred_classes[i], target_bboxes[i], target_classes[i]
                )
                for i in range(len(images))
            )
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/len(dataloader):.4f}")

In [52]:
# Check if MPS is available, otherwise fallback to CUDA or CPU
device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available() else "cpu"
)

device = "cpu"

# Initialize the model, loss, and optimizer
num_classes = 4
model = CustomObjectDetector(num_classes)
criterion = ObjectDetectionLoss()

# Move model to the selected device (MPS, CUDA, or CPU)
model.to(device)

# Get DataLoaders
train_loader, val_loader, test_loader = get_dataloaders(batch_size=16)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train the model
train(model, train_loader, optimizer, criterion, device, epochs=10)

RuntimeError: size mismatch (got input: [4], target: [0])